# Phase 2 — Model Training & Hyperparameter Tuning
### Om Sharma | Coding Club Recruitment

In [ ]:
!pip install xgboost scikit-learn joblib -q

import pandas as pd
import numpy as np
import joblib
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold
from sklearn.metrics import mean_squared_error
from scipy.stats import randint, uniform
from xgboost import XGBRegressor

RANDOM_STATE   = 42
REFERENCE_YEAR = 2016
TARGET         = 'sellingprice'

CATEGORICAL_COLS = ['make','model','trim','body','transmission','state','color','interior']
NUMERIC_COLS     = ['year','condition','odometer']

print('Setup complete')

## §1 Load & Clean Data

In [ ]:
df = pd.read_csv('car_auction_clean.csv')
print(f'Raw shape: {df.shape}')
df.head(3)

In [ ]:
df = df.dropna(subset=[TARGET])
df = df[(df[TARGET] >= 500) & (df[TARGET] <= 220_000)]

df['condition'] = df['condition'].fillna(df['condition'].median())
df['odometer']  = df['odometer'].fillna(df['odometer'].median())

for col in CATEGORICAL_COLS:
    df[col] = df[col].fillna('unknown').astype(str)

for col in ['condition', 'odometer']:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR    = Q3 - Q1
    df     = df[(df[col] >= Q1 - 1.5*IQR) & (df[col] <= Q3 + 1.5*IQR)]

print(f'Clean shape : {df.shape}')
print(f'Nulls       : {df.isnull().sum().sum()}')
print(f'Price range : ${df[TARGET].min():,.0f} — ${df[TARGET].max():,.0f}')
print(f'Price median: ${df[TARGET].median():,.0f}')

## §2 Feature Engineering

In [ ]:
df['age_years']         = (REFERENCE_YEAR - df['year']).clip(lower=1)
df['usage_intensity']   = df['odometer'] / df['age_years']
df['odo_per_condition'] = df['odometer'] / df['condition'].clip(lower=0.1)
df['cond_x_year']       = df['condition'] * df['year']

FEATURE_COLS = [
    'year', 'condition', 'odometer',
    'usage_intensity', 'age_years', 'odo_per_condition', 'cond_x_year',
    'make', 'model', 'trim', 'body', 'transmission', 'state', 'color', 'interior'
]

print('Engineered features:')
for f in ['usage_intensity','age_years','odo_per_condition','cond_x_year']:
    print(f'  {f}: {df[f].describe()["mean"]:.2f} mean')

## §3 Label Encoding & Train/Val Split

In [ ]:
encoders = {}
for col in CATEGORICAL_COLS:
    le        = LabelEncoder()
    df[col]   = le.fit_transform(df[col].astype(str))
    encoders[col] = le

joblib.dump(encoders, 'encoders_OmSharma.pkl')
print('Encoders saved: encoders_OmSharma.pkl')

X     = df[FEATURE_COLS].values.astype(float)
y_raw = df[TARGET].values
y     = np.log1p(y_raw)

mask = np.isfinite(y) & np.all(np.isfinite(X), axis=1)
X, y, y_raw = X[mask], y[mask], y_raw[mask]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
y_raw_val = np.expm1(y_val)

print(f'Train: {X_train.shape}  |  Val: {X_val.shape}')

## §4 Baseline Model (Default XGBoost)

In [ ]:
t0          = time.time()
xgb_default = XGBRegressor(n_estimators=500, random_state=RANDOM_STATE, verbosity=0)
xgb_default.fit(X_train, y_train)

preds_default = np.expm1(xgb_default.predict(X_val))
rmse_default  = np.sqrt(mean_squared_error(y_raw_val, preds_default))
mape_default  = np.mean(np.abs((y_raw_val - preds_default) / y_raw_val)) * 100

print(f'Default XGBoost')
print(f'  RMSE : ${rmse_default:,.0f}')
print(f'  MAPE : {mape_default:.2f}%')
print(f'  Time : {time.time()-t0:.1f}s')

## §5 Hyperparameter Tuning (RandomizedSearchCV)

In [ ]:
param_dist = {
    'n_estimators'     : randint(300, 1201),
    'max_depth'        : randint(3, 10),
    'learning_rate'    : uniform(0.01, 0.29),
    'subsample'        : uniform(0.60, 0.40),
    'colsample_bytree' : uniform(0.50, 0.50),
    'min_child_weight' : randint(1, 11),
    'gamma'            : uniform(0.0, 0.5),
    'reg_alpha'        : uniform(0.0, 1.0),
    'reg_lambda'       : uniform(0.5, 2.5),
}

xgb_base = XGBRegressor(
    random_state = RANDOM_STATE,
    verbosity    = 0,
    n_jobs       = -1,
    tree_method  = 'hist'
)

rscv = RandomizedSearchCV(
    estimator           = xgb_base,
    param_distributions = param_dist,
    n_iter              = 50,
    cv                  = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
    scoring             = 'neg_root_mean_squared_error',
    random_state        = RANDOM_STATE,
    n_jobs              = -1,
    verbose             = 1
)

t0 = time.time()
rscv.fit(X_train, y_train)
print(f'Tuning done in {time.time()-t0:.1f}s')
print(f'Best params: {rscv.best_params_}')

## §6 Tuned Model Evaluation & Comparison

In [ ]:
best_model    = rscv.best_estimator_
preds_tuned   = np.expm1(best_model.predict(X_val))
rmse_tuned    = np.sqrt(mean_squared_error(y_raw_val, preds_tuned))
mape_tuned    = np.mean(np.abs((y_raw_val - preds_tuned) / y_raw_val)) * 100
improvement   = (rmse_default - rmse_tuned) / rmse_default * 100

print('=' * 45)
print(f'{'Model':<25} {'RMSE':>8} {'MAPE':>8}')
print('-' * 45)
print(f'{'XGBoost (default)':<25} ${rmse_default:>7,.0f} {mape_default:>7.2f}%')
print(f'{'XGBoost (tuned)':<25} ${rmse_tuned:>7,.0f} {mape_tuned:>7.2f}%')
print('-' * 45)
print(f'Improvement: {improvement:.1f}%')
print('=' * 45)

## §7 Feature Importance

In [ ]:
import matplotlib.pyplot as plt

importances = best_model.feature_importances_
feat_df     = pd.DataFrame({'feature': FEATURE_COLS, 'importance': importances})
feat_df     = feat_df.sort_values('importance', ascending=True)

plt.figure(figsize=(10, 8))
plt.barh(feat_df['feature'], feat_df['importance'], color='steelblue')
plt.xlabel('Feature Importance (XGBoost)')
plt.title('Feature Importance — Tuned XGBoost')
plt.tight_layout()
plt.show()

print('\nTop 5 features:')
print(feat_df.tail(5).to_string(index=False))

## §8 Save Model

In [ ]:
joblib.dump(best_model, 'model_OmSharma.pkl')
print('Saved: model_OmSharma.pkl')

print('\nSample predictions:')
print(f'  {"Actual":>10}  {"Predicted":>10}  {"Error":>10}')
for i in range(8):
    err = preds_tuned[i] - y_raw_val[i]
    print(f'  ${y_raw_val[i]:>9,.0f}  ${preds_tuned[i]:>9,.0f}  ${err:>+9,.0f}')